# Transaction-level Retrospective Mining

依據 `strategy/transaction_level_retrospective_mining.md` 實作。

- 雙模式：Mode A（Exploratory）/ Mode B（Leakage-safe Transfer）
- Stage A/B/C 訓練比例
- 模型比較：Logistic / Decision Tree / LightGBM / XGBoost（可選）
- 指標：PR-AUC + threshold scan + risk tier


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

import matplotlib.pyplot as plt

HAS_LGB = True
HAS_XGB = True
try:
    import lightgbm as lgb
except Exception:
    HAS_LGB = False
try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGB = False


In [ ]:
# ==== Config ====
DATA_PATH = Path('../Transactions Data.csv')
OUT_DIR = Path('../')

MODE = 'B'  # 'A' retrospective exploratory | 'B' leakage-safe transfer
STAGE = 'A' # A/B/C
STAGE_TRAIN_FRAC = {'A':0.2, 'B':0.5, 'C':1.0}

USE_SAMPLE = True
SAMPLE_FRAC = 0.20
RANDOM_STATE = 42

TRAIN_RATIO = 0.70
VALID_RATIO = 0.15

RISK_T_MID = 0.10
RISK_T_HIGH = 0.30


In [ ]:
# ==== Load data ====
df = pd.read_csv(DATA_PATH)
if USE_SAMPLE:
    parts = []
    for _, g in df.groupby('step', sort=False):
        if len(g) <= 1:
            parts.append(g)
        else:
            parts.append(g.sample(frac=min(SAMPLE_FRAC, 1.0), random_state=RANDOM_STATE))
    df = pd.concat(parts, ignore_index=True)

df = df.sort_values('step').reset_index(drop=True)
print(df.shape, 'fraud rate=', df['isFraud'].mean())


In [ ]:
# ==== Base features ====
work = df.copy()
work['log1p_amount'] = np.log1p(work['amount'].clip(lower=0))
work['deltaOrig'] = work['oldbalanceOrg'] - work['newbalanceOrig']
work['deltaDest'] = work['newbalanceDest'] - work['oldbalanceDest']
work['balance_inconsistency_flag'] = ((work['oldbalanceOrg'] < work['newbalanceOrig']) | (work['newbalanceDest'] < work['oldbalanceDest'])).astype(int)
TARGET = 'isFraud'


## Feature generation by mode
- Mode A：可用全歷史聚合
- Mode B：逐筆 past-only 生成（leakage-safe）

In [ ]:
def build_mode_a_features(df):
    out = df.copy()

    # account aggregates (full-history, retrospective)
    orig_agg = out.groupby('nameOrig').agg(
        orig_out_degree=('amount','size'),
        orig_out_amount_sum=('amount','sum'),
        orig_out_amount_mean=('amount','mean'),
        orig_out_amount_std=('amount','std'),
        orig_unique_dest=('nameDest','nunique')
    ).fillna(0)

    dest_agg = out.groupby('nameDest').agg(
        dest_in_degree=('amount','size'),
        dest_in_amount_sum=('amount','sum'),
        dest_in_amount_mean=('amount','mean'),
        dest_in_amount_std=('amount','std'),
        dest_unique_orig=('nameOrig','nunique')
    ).fillna(0)

    pair_agg = out.groupby(['nameOrig','nameDest']).agg(
        pair_txn_count=('amount','size'),
        pair_amount_sum=('amount','sum'),
        pair_amount_mean=('amount','mean')
    ).fillna(0)

    out = out.join(orig_agg, on='nameOrig')
    out = out.join(dest_agg, on='nameDest')
    out = out.join(pair_agg, on=['nameOrig','nameDest'])

    # derived suspicious ratios
    eps = 1e-9
    out['orig_repeat_counterparty_ratio'] = 1 - (out['orig_unique_dest'] / out['orig_out_degree'].clip(lower=1))
    out['dest_repeat_counterparty_ratio'] = 1 - (out['dest_unique_orig'] / out['dest_in_degree'].clip(lower=1))
    out['amount_vs_orig_mean_ratio'] = out['amount'] / (out['orig_out_amount_mean'].abs() + eps)
    out['amount_vs_pair_mean_ratio'] = out['amount'] / (out['pair_amount_mean'].abs() + eps)
    out['orig_dest_degree_ratio'] = out['orig_out_degree'] / out['dest_in_degree'].clip(lower=1)

    # exploratory proxy (non-causal): mark first-time pair by global rank
    out['pair_rank_global'] = out.groupby(['nameOrig','nameDest']).cumcount() + 1
    out['is_first_time_pair'] = (out['pair_rank_global'] == 1).astype(int)

    # large amount threshold (global quantile, retrospective-only)
    q95 = out['amount'].quantile(0.95)
    out['new_counterparty_and_large_amount'] = ((out['is_first_time_pair']==1) & (out['amount']>=q95)).astype(int)
    out['burst_and_large'] = ((out['orig_out_degree']>=out['orig_out_degree'].quantile(0.9)) & (out['amount']>=q95)).astype(int)

    return out


def build_mode_b_features(df):
    out = df.copy()
    n = len(out)

    orig = out['nameOrig'].values
    dest = out['nameDest'].values
    amt = out['amount'].values.astype(float)
    step = out['step'].values.astype(int)

    out_deg = defaultdict(int); in_deg = defaultdict(int)
    out_sum = defaultdict(float); in_sum = defaultdict(float)
    out_sum_sq = defaultdict(float); in_sum_sq = defaultdict(float)
    out_last = defaultdict(lambda: -1); in_last = defaultdict(lambda: -1)
    orig_to_dest = defaultdict(set); dest_from_orig = defaultdict(set)
    pair_cnt = defaultdict(int); pair_sum = defaultdict(float); pair_last = defaultdict(lambda: -1)
    orig_dest_cnt = defaultdict(lambda: defaultdict(int))

    f = {k: np.zeros(n, dtype=float) for k in [
        'orig_out_degree','dest_in_degree','orig_unique_dest','dest_unique_orig',
        'orig_out_amount_sum','dest_in_amount_sum','orig_out_amount_mean','dest_in_amount_mean',
        'orig_out_amount_std','dest_in_amount_std','orig_recent_gap','dest_recent_gap',
        'pair_txn_count','pair_amount_sum','pair_amount_mean','pair_last_seen_gap','is_first_time_pair',
        'orig_repeat_counterparty_ratio','dest_repeat_counterparty_ratio',
        'amount_vs_orig_mean_ratio','amount_vs_pair_mean_ratio','orig_dest_degree_ratio',
        'new_counterparty_and_large_amount','burst_and_large'
    ]}

    eps = 1e-9
    for i in range(n):
        o, d, a, s = orig[i], dest[i], amt[i], max(step[i],1)
        p = (o,d)

        od, dd = out_deg[o], in_deg[d]
        ous, dis = out_sum[o], in_sum[d]
        ousq, disq = out_sum_sq[o], in_sum_sq[d]

        ostd = np.sqrt(max(0.0, ousq/max(od,1) - (ous/max(od,1))**2))
        dstd = np.sqrt(max(0.0, disq/max(dd,1) - (dis/max(dd,1))**2))

        ouniq = len(orig_to_dest[o]); duniq = len(dest_from_orig[d])
        pc, ps = pair_cnt[p], pair_sum[p]

        f['orig_out_degree'][i]=od; f['dest_in_degree'][i]=dd
        f['orig_unique_dest'][i]=ouniq; f['dest_unique_orig'][i]=duniq
        f['orig_out_amount_sum'][i]=ous; f['dest_in_amount_sum'][i]=dis
        f['orig_out_amount_mean'][i]=ous/max(od,1); f['dest_in_amount_mean'][i]=dis/max(dd,1)
        f['orig_out_amount_std'][i]=ostd; f['dest_in_amount_std'][i]=dstd
        f['orig_recent_gap'][i]=0 if out_last[o]<0 else s-out_last[o]
        f['dest_recent_gap'][i]=0 if in_last[d]<0 else s-in_last[d]
        f['pair_txn_count'][i]=pc; f['pair_amount_sum'][i]=ps; f['pair_amount_mean'][i]=ps/max(pc,1)
        f['pair_last_seen_gap'][i]=0 if pair_last[p]<0 else s-pair_last[p]
        f['is_first_time_pair'][i]=1.0 if pc==0 else 0.0

        f['orig_repeat_counterparty_ratio'][i]=1 - (ouniq/max(od,1))
        f['dest_repeat_counterparty_ratio'][i]=1 - (duniq/max(dd,1))
        f['amount_vs_orig_mean_ratio'][i]=a/(f['orig_out_amount_mean'][i]+eps)
        f['amount_vs_pair_mean_ratio'][i]=a/(f['pair_amount_mean'][i]+eps)
        f['orig_dest_degree_ratio'][i]=od/max(dd,1)

        # causal proxies
        large_proxy = a > (f['orig_out_amount_mean'][i] + 2*max(f['orig_out_amount_std'][i],1e-6))
        f['new_counterparty_and_large_amount'][i] = 1.0 if (pc==0 and large_proxy) else 0.0
        burst_proxy = od/max(s,1) > 0.02
        f['burst_and_large'][i] = 1.0 if (burst_proxy and large_proxy) else 0.0

        # update
        out_deg[o]+=1; in_deg[d]+=1
        out_sum[o]+=a; in_sum[d]+=a
        out_sum_sq[o]+=a*a; in_sum_sq[d]+=a*a
        out_last[o]=s; in_last[d]=s
        orig_to_dest[o].add(d); dest_from_orig[d].add(o)
        pair_cnt[p]+=1; pair_sum[p]+=a; pair_last[p]=s
        orig_dest_cnt[o][d]+=1

    for k,v in f.items():
        out[k]=v
    return out


In [ ]:
# ==== Build selected mode features ====
if MODE == 'A':
    work = build_mode_a_features(work)
else:
    work = build_mode_b_features(work)

print('Mode', MODE, 'feature table shape:', work.shape)


In [ ]:
# ==== Split and stage subset ====
steps = np.sort(work['step'].unique())
train_cut = int(len(steps)*TRAIN_RATIO)
valid_cut = int(len(steps)*(TRAIN_RATIO+VALID_RATIO))
train_max = steps[train_cut-1]
valid_max = steps[valid_cut-1]

train_mask_full = work['step'] <= train_max
valid_mask = (work['step'] > train_max) & (work['step'] <= valid_max)
test_mask = work['step'] > valid_max

train_frac = STAGE_TRAIN_FRAC[STAGE]

X_all = work.copy(); y_all = work[TARGET].values
X_train_full = X_all.loc[train_mask_full].copy(); y_train_full = y_all[train_mask_full]
X_valid = X_all.loc[valid_mask].copy(); y_valid = y_all[valid_mask]
X_test = X_all.loc[test_mask].copy(); y_test = y_all[test_mask]

if train_frac < 1.0:
    tmp = X_train_full.copy(); tmp[TARGET] = y_train_full
    tmp = tmp.groupby(TARGET, group_keys=False).sample(frac=train_frac, random_state=RANDOM_STATE).sample(frac=1.0, random_state=RANDOM_STATE)
    X_train = tmp.drop(columns=[TARGET]).copy(); y_train = tmp[TARGET].values
else:
    X_train = X_train_full.copy(); y_train = y_train_full.copy()

print('Stage', STAGE, 'train_frac', train_frac)
print('train', X_train.shape, y_train.mean(), 'valid', X_valid.shape, y_valid.mean(), 'test', X_test.shape, y_test.mean())


In [ ]:
# ==== Modeling feature columns ====
base_num = ['step','amount','log1p_amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','deltaOrig','deltaDest','balance_inconsistency_flag','isFlaggedFraud']
base_cat = ['type']

retro_num = [c for c in work.columns if c not in set(['nameOrig','nameDest',TARGET,'type']) and c not in base_num and c!='step']
# keep step in base_num already

feature_cols = base_num + base_cat + retro_num
X_train = X_train[feature_cols].copy(); X_valid = X_valid[feature_cols].copy(); X_test = X_test[feature_cols].copy()

print('features:', len(feature_cols), 'retro:', len(retro_num))


In [ ]:
# ==== Eval helpers ====
def eval_prob(y, p, name):
    return {'model':name, 'pr_auc':average_precision_score(y,p), 'roc_auc':roc_auc_score(y,p)}

def threshold_scan(y,p,name,ths=None):
    if ths is None:
        ths=[0.01,0.02,0.05,0.1,0.15,0.2,0.3,0.5]
    rows=[]
    y=np.asarray(y)
    for t in ths:
        pr=(p>=t).astype(int)
        tp=((y==1)&(pr==1)).sum(); fp=((y==0)&(pr==1)).sum(); fn=((y==1)&(pr==0)).sum()
        prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1); f1=2*prec*rec/max(prec+rec,1e-12)
        rows.append({'model':name,'threshold':t,'precision':prec,'recall':rec,'f1':f1,'pred_pos':int((pr==1).sum())})
    return pd.DataFrame(rows)

def risk_tier(y,p,name,t_mid=0.10,t_high=0.30):
    d=pd.DataFrame({'y':y,'p':p})
    d['tier']=pd.cut(d['p'],[-np.inf,t_mid,t_high,np.inf],labels=['Low','Medium','High'])
    o=d.groupby('tier',observed=True).agg(volume=('y','size'),fraud_count=('y','sum'),fraud_rate=('y','mean')).reset_index()
    o['model']=name
    return o[['model','tier','volume','fraud_count','fraud_rate']]


In [ ]:
# ==== Train models and compare ====
metrics=[]; scans=[]; tiers=[]; pred={}

pre_lr = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), base_num + retro_num),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]), base_cat),
])
pre_tree = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median'))]), base_num + retro_num),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]), base_cat),
])

# Logistic
m = Pipeline([('prep',pre_lr),('clf',LogisticRegression(max_iter=1000,class_weight='balanced',solver='saga',n_jobs=-1,random_state=RANDOM_STATE))])
m.fit(X_train,y_train); p = m.predict_proba(X_valid)[:,1]
metrics.append(eval_prob(y_valid,p,'Logistic')); scans.append(threshold_scan(y_valid,p,'Logistic')); tiers.append(risk_tier(y_valid,p,'Logistic',RISK_T_MID,RISK_T_HIGH)); pred['Logistic']=p

# Decision Tree
m = Pipeline([('prep',pre_tree),('clf',DecisionTreeClassifier(max_depth=10,min_samples_leaf=200,class_weight='balanced',random_state=RANDOM_STATE))])
m.fit(X_train,y_train); p = m.predict_proba(X_valid)[:,1]
metrics.append(eval_prob(y_valid,p,'DecisionTree')); scans.append(threshold_scan(y_valid,p,'DecisionTree')); tiers.append(risk_tier(y_valid,p,'DecisionTree',RISK_T_MID,RISK_T_HIGH)); pred['DecisionTree']=p

# LightGBM
if HAS_LGB:
    xtr=X_train.copy(); xva=X_valid.copy(); xtr['type']=xtr['type'].astype('category'); xva['type']=xva['type'].astype('category')
    neg=(y_train==0).sum(); pos=(y_train==1).sum(); spw=max(1.0,neg/max(pos,1))
    m=lgb.LGBMClassifier(objective='binary',n_estimators=400,learning_rate=0.05,num_leaves=64,min_child_samples=100,subsample=0.8,colsample_bytree=0.8,scale_pos_weight=spw,random_state=RANDOM_STATE,n_jobs=-1)
    m.fit(xtr,y_train,eval_set=[(xva,y_valid)],eval_metric='aucpr',categorical_feature=['type'],callbacks=[lgb.early_stopping(50,verbose=False)])
    p=m.predict_proba(xva)[:,1]
    metrics.append(eval_prob(y_valid,p,'LightGBM')); scans.append(threshold_scan(y_valid,p,'LightGBM')); tiers.append(risk_tier(y_valid,p,'LightGBM',RISK_T_MID,RISK_T_HIGH)); pred['LightGBM']=p

# XGBoost optional
if HAS_XGB:
    xtr=pd.get_dummies(X_train.copy(),columns=['type'],drop_first=False)
    xva=pd.get_dummies(X_valid.copy(),columns=['type'],drop_first=False).reindex(columns=xtr.columns,fill_value=0)
    neg=(y_train==0).sum(); pos=(y_train==1).sum(); spw=max(1.0,neg/max(pos,1))
    m=XGBClassifier(objective='binary:logistic',eval_metric='aucpr',n_estimators=300,learning_rate=0.05,max_depth=6,min_child_weight=5,subsample=0.8,colsample_bytree=0.8,scale_pos_weight=spw,random_state=RANDOM_STATE,n_jobs=-1,tree_method='hist')
    m.fit(xtr,y_train,eval_set=[(xva,y_valid)],verbose=False)
    p=m.predict_proba(xva)[:,1]
    metrics.append(eval_prob(y_valid,p,'XGBoost')); scans.append(threshold_scan(y_valid,p,'XGBoost')); tiers.append(risk_tier(y_valid,p,'XGBoost',RISK_T_MID,RISK_T_HIGH)); pred['XGBoost']=p

metrics_df = pd.DataFrame(metrics).sort_values('pr_auc',ascending=False)
scan_df = pd.concat(scans,ignore_index=True)
tier_df = pd.concat(tiers,ignore_index=True)
metrics_df


In [ ]:
# ==== Export outputs ====
OUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(OUT_DIR/'transaction_pattern_model_importance.csv', index=False)
scan_df.to_csv(OUT_DIR/'transaction_pattern_threshold_scan.csv', index=False)
tier_df.to_csv(OUT_DIR/'transaction_pattern_risk_tier_summary.csv', index=False)
work.to_csv(OUT_DIR/'transaction_feature_table.csv', index=False)

with open(OUT_DIR/'transaction_pattern_findings.md','w',encoding='utf-8') as f:
    f.write('# Transaction Pattern Findings (Auto)\n\n')
    f.write(f'- Mode: {MODE}\n- Stage: {STAGE}\n\n')
    f.write('## Model metrics\n')
    f.write(metrics_df.to_markdown(index=False))

print('Saved outputs in project root')


## 使用建議

- 先用 Mode A 找高關聯特徵。
- 再把候選特徵改寫/切換為 Mode B，做可遷移 baseline 比較。
